# NB2 — 사람 metric depth (Sapiens 1B) **[GPU]**
기존 검증 노트북 `person_depth_pipeline_VIDEO_v3_chunked` 의 per-frame 로직 (Seg → 안정 bbox → crop → Depth → LiDAR anchor → tail 제거 + Normal refine → full-frame) 을 **그대로** 사용.
바뀐 것 (colab_kit):
- 입력: zip 업로드 대신 NB1 의 `00_extract/` 를 직접 읽음 (key = `WALK_side_{fid}`)
- 작업 폴더 `work/` 가 **드라이브** (`sessions/{SESSION}/01_depth/work`) → 끊겨도 보존, 이미 처리한 프레임은 건너뜀
- 모델 파일은 `/content/models` (드라이브 용량 절약, 세션마다 재다운로드 ~수 분)
- mask/crop 디버그 png 는 `SAVE_DEBUG_EVERY` 프레임마다만 저장

런타임: **GPU (T4 이상)**. 라이선스 Sapiens CC BY-NC 4.0.
**끊겼을 때**: 0 → 1 → 2 → 3 → (5 대신 **5-B 캐시 로드**) → 5.5 → 6 → 7 → 8 순서로 다시 실행.

In [ ]:
#@title 0. 설정 — 세션마다 여기만 바꾸면 됨
ROOT    = "/content/drive/MyDrive/walking_0729"   #@param {type:"string"}
SESSION = "0729_walk_3kmh"                        #@param {type:"string"}

from google.colab import drive
drive.mount("/content/drive")
import os, sys, importlib
os.environ["WALK_ROOT"] = ROOT
os.environ["WALK_SESSION"] = SESSION
sys.path.insert(0, f"{ROOT}/scripts")
import walkcfg; importlib.reload(walkcfg)
assert os.path.isdir(f"{ROOT}/scripts"), f"{ROOT}/scripts 없음 — colab_kit 업로드 위치 확인"
print("세션 폴더:", walkcfg.SESS)

In [ ]:
# 1. 환경
import torch
print("CUDA:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
print("torch:", torch.__version__)

In [ ]:
#@title 2. 입력 = NB1 00_extract (zip 불필요)
import os, glob, json
os.makedirs(walkcfg.DEPTH_BASE, exist_ok=True)
os.chdir(walkcfg.DEPTH_BASE)          # 이후 상대경로 work/, inputs/ 는 드라이브에 저장
MODEL_DIR = "/content/models"
E = walkcfg.EXTRACT_DIR
K = json.load(open(f"{E}/intrinsics.json"))
INTR = {"_default": {k: K[k] for k in ("fx", "fy", "cx", "cy")}}
os.makedirs("inputs", exist_ok=True)
json.dump(INTR, open("inputs/intrinsics.json", "w"), indent=2)   # phase4_reanchor 가 읽음

FRAMES = []
for p in sorted(glob.glob(f"{E}/rgb/*.jpg")):
    fid = os.path.basename(p)[:-4]
    d, c = f"{E}/depth_npy/{fid}.npy", f"{E}/conf_npy/{fid}.npy"
    FRAMES.append(dict(key=f"WALK_side_{fid}", pose="WALK", view="side", fid=fid, rgb=p,
                       depth=d if os.path.exists(d) else None, conf=c if os.path.exists(c) else None))
def get_intr(key, W, H):
    d = INTR["_default"]; return d["fx"], d["fy"], d["cx"], d["cy"]
SUBJECT_HEIGHT_M = 1.81

CHUNK = 120            # T4 시스템 RAM 안전선 (원본 노트북 값)
SAVE_NORMAL = True     # phase4 재-refine 이 normal 사용 → True 유지
SAVE_DEBUG_EVERY = 25  # mask/crop png 저장 간격
def _dbg(f): return int(f["fid"]) % SAVE_DEBUG_EVERY == 0
for _d in ["work/mask", "work/crop", "work/out", "work/video", "work/analysis"]:
    os.makedirs(_d, exist_ok=True)
assert FRAMES, "00_extract/rgb 비어있음 → NB1 먼저"
print(f"프레임 {len(FRAMES)} | fid {FRAMES[0]['fid']}~{FRAMES[-1]['fid']} | depth {sum(bool(f['depth']) for f in FRAMES)} | conf {sum(bool(f['conf']) for f in FRAMES)}")
print("K:", INTR["_default"])

In [ ]:
# 4. 공용 함수
import os
import subprocess
import numpy as np
import cv2
import torch


def ensure_model(url, path, min_gb=3.0):
    """
    정상 크기의 모델 파일이 존재하면 다운로드를 생략.
    작은 파일은 다운로드 실패 파일로 보고 삭제 후 다시 받음.
    """

    min_bytes = int(min_gb * 1024**3)

    if os.path.exists(path):
        size = os.path.getsize(path)

        if size >= min_bytes:
            print(
                f"✓ 모델 존재 → 다운로드 생략\n"
                f"  {path} ({size / 1024**3:.2f} GB)"
            )
            return

        print(
            f"⚠ 불완전 모델 파일 삭제\n"
            f"  {path} ({size / 1024**2:.1f} MB)"
        )
        os.remove(path)

    temp_path = path + ".part"

    if os.path.exists(temp_path):
        os.remove(temp_path)

    print("다운로드 시작:", os.path.basename(path))

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            "--tries=3",
            "--timeout=30",
            "-O",
            temp_path,
            url,
        ],
        check=True,
    )

    downloaded_size = (
        os.path.getsize(temp_path)
        if os.path.exists(temp_path)
        else 0
    )

    if downloaded_size < min_bytes:
        if os.path.exists(temp_path):
            os.remove(temp_path)

        raise RuntimeError(
            f"모델 다운로드 실패 또는 불완전 파일\n"
            f"받은 용량: {downloaded_size / 1024**2:.1f} MB\n"
            f"URL: {url}"
        )

    os.replace(temp_path, path)

    print(
        f"✓ 다운로드 완료\n"
        f"  {path} ({downloaded_size / 1024**3:.2f} GB)"
    )


def load_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(path)

    # RGBA PNG가 있으면 alpha를 전신 마스크로 우선 사용
    if img.ndim == 3 and img.shape[2] == 4:
        rgb = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2RGB)
        alpha = img[:, :, 3] > 128
        return rgb, alpha

    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), None


def to_rgb_res(a, W, H, nearest=True):
    if a is None or a.shape[:2] == (H, W):
        return a

    return cv2.resize(
        a,
        (W, H),
        interpolation=(
            cv2.INTER_NEAREST
            if nearest
            else cv2.INTER_LINEAR
        ),
    )


def pad_bbox_3_4(x0, y0, x1, y1, margin=0.18, ratio=0.75):
    bw = max(1, x1 - x0)
    bh = max(1, y1 - y0)

    cx = (x0 + x1) / 2
    cy = (y0 + y1) / 2

    bw *= 1 + margin
    bh *= 1 + margin

    if bw / bh > ratio:
        bh = bw / ratio
    else:
        bw = bh * ratio

    return (
        cx - bw / 2,
        cy - bh / 2,
        cx + bw / 2,
        cy + bh / 2,
    )


def crop_pad(arr, box, fill=0):
    x0, y0, x1, y1 = [int(round(v)) for v in box]

    cw = max(1, x1 - x0)
    ch = max(1, y1 - y0)

    if arr.ndim == 3:
        out = np.full(
            (ch, cw, arr.shape[2]),
            fill,
            arr.dtype,
        )
    else:
        out = np.full(
            (ch, cw),
            fill,
            arr.dtype,
        )

    H, W = arr.shape[:2]

    sx0 = max(0, x0)
    sy0 = max(0, y0)
    sx1 = min(W, x1)
    sy1 = min(H, y1)

    if sx1 > sx0 and sy1 > sy0:
        dx = sx0 - x0
        dy = sy0 - y0

        out[
            dy:dy + (sy1 - sy0),
            dx:dx + (sx1 - sx0),
        ] = arr[sy0:sy1, sx0:sx1]

    return out, (x0, y0, x1, y1)


def ransac_lin(x, y, iters=3000, k=3.0, seed=0):
    rng = np.random.default_rng(seed)

    best_inlier = None
    best_count = -1
    n = len(x)

    for _ in range(iters):
        i, j = rng.integers(0, n, 2)

        dx = x[i] - x[j]

        if abs(dx) < 1e-9:
            continue

        a = (y[i] - y[j]) / dx
        b = y[i] - a * x[i]

        residual = np.abs(y - (a * x + b))
        mad = (
            np.median(
                np.abs(
                    residual - np.median(residual)
                )
            )
            * 1.4826
            + 1e-12
        )

        inlier = residual < k * mad
        count = int(inlier.sum())

        if count > best_count:
            best_count = count
            best_inlier = inlier

    if best_inlier is None or best_inlier.sum() < 2:
        raise RuntimeError("RANSAC 선형 적합 실패")

    for _ in range(2):
        A = np.stack(
            [
                x[best_inlier],
                np.ones(int(best_inlier.sum())),
            ],
            axis=1,
        )

        a, b = np.linalg.lstsq(
            A,
            y[best_inlier],
            rcond=None,
        )[0]

        residual = np.abs(y - (a * x + b))
        mad = (
            np.median(
                np.abs(
                    residual - np.median(residual)
                )
            )
            * 1.4826
            + 1e-12
        )

        best_inlier = residual < k * mad

    return float(a), float(b), best_inlier


def anchor_auto(s, z, force_mode=None):
    # depth-mode 적합 (z = a*s + b)
    aA, bA, iA = ransac_lin(s, z)
    relA = np.median(
        np.abs((aA * s + bA) - z)
        / np.clip(z, 1e-6, None)
    )
    depth_res = dict(
        mode="depth",
        a=float(aA),
        b=float(bA),
        rel=float(relA),
        inlier=float(iA.mean()),
    )
    # [VIDEO v2] Sapiens depth-1B 출력은 항상 depth-like (refit_a≈1, 전 프레임 양수 검증).
    #   force_mode="depth" 면 disparity 분기를 타지 않아 결정-경계 떨림(mode flip)을 원천 제거.
    if force_mode == "depth":
        return depth_res

    # disparity-mode 적합 (1/z = a*s + b)
    aB, bB, iB = ransac_lin(
        s,
        1.0 / np.clip(z, 1e-6, None),
    )
    zB = 1.0 / np.clip(
        aB * s + bB,
        1e-6,
        None,
    )
    relB = np.median(
        np.abs(zB - z)
        / np.clip(z, 1e-6, None)
    )
    disp_res = dict(
        mode="disparity",
        a=float(aB),
        b=float(bB),
        rel=float(relB),
        inlier=float(iB.mean()),
    )
    if force_mode == "disparity":
        return disp_res

    return depth_res if relA <= relB else disp_res


def apply_anchor(s, p):
    if p["mode"] == "depth":
        return p["a"] * s + p["b"]

    return 1.0 / np.clip(
        p["a"] * s + p["b"],
        1e-6,
        None,
    )


def measure_height_m(z, mask, fx, fy, cx, cy):
    v, u = np.where(
        mask
        & (z > 0)
        & np.isfinite(z)
    )

    if len(v) < 50:
        return None

    Y = (v - cy) * z[v, u] / fy

    return float(
        np.percentile(Y, 99.5)
        - np.percentile(Y, 0.5)
    )


def measure_width_m(
    z,
    mask,
    fx,
    fy,
    cx,
    cy,
    band=(0.10, 0.28),
):
    v, u = np.where(
        mask
        & (z > 0)
        & np.isfinite(z)
    )

    if len(v) < 50:
        return None

    vt = v.min()
    vb = v.max()

    lo = vt + band[0] * (vb - vt)
    hi = vt + band[1] * (vb - vt)

    sel = (v >= lo) & (v <= hi)

    if sel.sum() < 20:
        return None

    X = (
        (u[sel] - cx)
        * z[v[sel], u[sel]]
        / fx
    )

    return float(
        np.percentile(X, 99)
        - np.percentile(X, 1)
    )


def unwrap_tensor(out):
    if isinstance(out, (list, tuple)):
        out = out[0]

    if isinstance(out, dict):
        out = next(iter(out.values()))

    if not torch.is_tensor(out):
        raise TypeError(
            f"모델 출력이 tensor가 아닙니다: {type(out)}"
        )

    return out


print("함수 OK")

## 5. Sapiens Seg 1B (pass-1, 전 프레임) — 끊긴 뒤 재시작이면 이 셀 대신 **5-B**

In [ ]:
# 5. Sapiens Seg 1B 전신 마스크 + bbox (RGBA alpha 있으면 우선, 없으면 Seg 1B)
import os, cv2, numpy as np, torch

SEG_REPO = "facebook/sapiens-seg-1b-torchscript"
SEG_FILE = "sapiens_1b_goliath_best_goliath_mIoU_7994_epoch_151_torchscript.pt2"
SEG_URL  = f"https://huggingface.co/{SEG_REPO}/resolve/main/{SEG_FILE}"
SEG_PATH = f"{MODEL_DIR}/sapiens_seg_1b.pt2"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs("work/mask", exist_ok=True)
ensure_model(SEG_URL, SEG_PATH, min_gb=3.5)
seg_model = torch.jit.load(SEG_PATH).cuda().eval()

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
HS, WS = 1024, 768


@torch.no_grad()
def sapiens_seg(rgb):
    """반환: label(원본 크기 class map), mask(background 제외 전신)."""
    H, W = rgb.shape[:2]
    x = cv2.resize(rgb, (WS, HS), interpolation=cv2.INTER_LINEAR)
    x = (x.astype(np.float32) / 255.0 - MEAN) / STD
    x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).float().cuda()
    logits = unwrap_tensor(seg_model(x))
    if logits.ndim != 4:
        raise RuntimeError(f"Seg 출력 shape 오류: {tuple(logits.shape)}")
    label = logits.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
    label = cv2.resize(label, (W, H), interpolation=cv2.INTER_NEAREST)
    mask = label > 0          # background=0, 사람 신체/의류/팔/다리/신발=1 이상
    return label, mask


def keep_main_person(mask):
    """작은 오검출 제거, 화면 중앙에 가까운 가장 큰 사람 성분 하나만 유지."""
    H, W = mask.shape
    m = mask.astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((7, 7), np.uint8), iterations=1)
    count, labels, stats, centers = cv2.connectedComponentsWithStats(m, connectivity=8)
    if count <= 1:
        return m.astype(bool)
    image_center = np.array([W / 2, H / 2], dtype=np.float32)
    best_id, best_score = None, -np.inf
    for component_id in range(1, count):
        area = int(stats[component_id, cv2.CC_STAT_AREA])
        if area < 0.002 * H * W:                      # 전체의 0.2% 미만 성분 제외
            continue
        dist = np.linalg.norm(centers[component_id] - image_center)
        score = area - 0.30 * (dist ** 2)             # 면적 클수록·중앙일수록 좋음
        if score > best_score:
            best_score, best_id = score, component_id
    if best_id is None:
        best_id = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    out = (labels == best_id).astype(np.uint8)
    out = cv2.morphologyEx(out, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8), iterations=1)
    return out.astype(bool)


MASKS, SKIPPED = {}, []
for f in FRAMES:
    rgb, alpha = load_rgb(f["rgb"])
    H, W = rgb.shape[:2]
    if alpha is not None:
        mask = keep_main_person(alpha)
        mask_source = "RGBA alpha"
    else:
        label, raw_mask = sapiens_seg(rgb)
        mask = keep_main_person(raw_mask)
        mask_source = "Sapiens Seg 1B"
        if _dbg(f): cv2.imwrite(f"work/mask/{f['key']}_seg_label.png", label)

    pct = 100.0 * mask.sum() / mask.size
    if mask.sum() < 0.005 * H * W:                    # 사람이 화면 밖 -> 죽지 말고 skip
        SKIPPED.append(f["key"])
        print(f"[SKIP] {f['key']} | {mask_source} | person={pct:.1f}% -> 제외(화면 밖)")
        continue

    ys, xs = np.where(mask)
    bbox = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
    MASKS[f["key"]] = mask
    f["bbox"], f["hw"] = bbox, (H, W)

    overlay = rgb.copy()
    overlay[mask] = (0.50 * overlay[mask] + 0.50 * np.array([0, 180, 0])).astype(np.uint8)
    x0, y0, x1, y1 = bbox
    cv2.rectangle(overlay, (x0, y0), (x1 - 1, y1 - 1), (255, 220, 0), 3)
    if _dbg(f): cv2.imwrite(f"work/mask/{f['key']}_maskvis.png", cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
    print(f"{f['key']} | {mask_source} | person={pct:.1f}% | bbox={bbox}")

FRAMES = [f for f in FRAMES if f["key"] in MASKS]     # skip 프레임 제외
del seg_model
torch.cuda.empty_cache()

print(f"\n✓ Seg 1B 완료 | 유효 {len(FRAMES)} / skip {len(SKIPPED)}")
print("work/mask/*_maskvis.png에서 전신이 잡혔는지 먼저 확인")

In [ ]:
#@title 5-A. pass-1 캐시 저장 (마스크 + bbox → 재시작 시 Seg 생략용)
import numpy as np
np.savez_compressed("work/analysis/seg_pass1.npz",
    keys=np.array([f["key"] for f in FRAMES]),
    bbox=np.array([f["bbox"] for f in FRAMES]), hw=np.array([f["hw"] for f in FRAMES]),
    masks=np.stack([np.packbits(MASKS[f["key"]]) for f in FRAMES]), shape=np.array(FRAMES[0]["hw"]))
print("saved work/analysis/seg_pass1.npz")

In [ ]:
#@title 5-B. (재시작 시에만) pass-1 캐시 로드
import numpy as np
c = np.load("work/analysis/seg_pass1.npz")
H, W = c["shape"]; byk = {k: i for i, k in enumerate(c["keys"])}
MASKS, keep = {}, []
for f in FRAMES:
    i = byk.get(f["key"])
    if i is None: continue
    MASKS[f["key"]] = np.unpackbits(c["masks"][i])[:H*W].reshape(H, W).astype(bool)
    f["bbox"] = tuple(int(v) for v in c["bbox"][i]); f["hw"] = tuple(int(v) for v in c["hw"][i]); keep.append(f)
FRAMES = keep
print("캐시 로드:", len(FRAMES), "프레임")

## 5.5 [VIDEO] 안정 bbox
mask떨림 → bbox떨림 → crop framing 이동 → Sapiens 입력 변동 → **상대 depth flicker**. 제자리 보행은 bbox가 거의 고정이므로 전 프레임 공통 robust bbox 1개로 crop을 고정해 flicker를 **소스에서** 차단. (Cell 5 직후 / Cell 6 직전 실행)

In [ ]:
# 5.5 [VIDEO] 안정 bbox -- 전 프레임 공통 (트레드밀 제자리 보행)
# 정적 4뷰로 쓰려면 USE_STABLE_BBOX=False 로 원래 per-frame bbox 동작.
import numpy as np

USE_STABLE_BBOX = True
MOVING          = False     # 피사체가 프레임을 크게 가로지르면 True(이동평균 bbox)
PCT_LO, PCT_HI  = 5, 95     # outlier 프레임(마스크 실패) 제거 percentile
SMOOTH_WIN      = 15        # MOVING일 때 시간 윈도우(30fps -> 0.5s)

if USE_STABLE_BBOX:
    for f in FRAMES:
        f["bbox_tight"] = f["bbox"]                     # 원본 tight 보관(커버리지 검증용)
    boxes = np.array([f["bbox"] for f in FRAMES], np.float64)
    if not MOVING:
        fixed = (int(np.percentile(boxes[:, 0], PCT_LO)), int(np.percentile(boxes[:, 1], PCT_LO)),
                 int(np.percentile(boxes[:, 2], PCT_HI)), int(np.percentile(boxes[:, 3], PCT_HI)))
        for f in FRAMES:
            f["bbox"] = fixed
        over = sum(1 for f in FRAMES
                   if f["bbox_tight"][0] < fixed[0] or f["bbox_tight"][1] < fixed[1]
                   or f["bbox_tight"][2] > fixed[2] or f["bbox_tight"][3] > fixed[3])
        print(f"[VIDEO] 공통 고정 bbox = {fixed}")
        print(f"[VIDEO] 공통 bbox 밖으로 살짝 벗어나는 프레임 {over}/{len(FRAMES)} (Cell 6 margin이 흡수)")
    else:
        k = np.ones(SMOOTH_WIN) / SMOOTH_WIN
        sm = np.stack([np.convolve(boxes[:, j], k, mode="same") for j in range(4)], axis=1)
        for i, f in enumerate(FRAMES):
            f["bbox"] = tuple(int(v) for v in sm[i])
        print(f"[VIDEO] 이동평균 bbox 적용 (win={SMOOTH_WIN})")
else:
    print("[VIDEO] USE_STABLE_BBOX=False -> per-frame bbox 유지(정적 동작)")

## 6~10 [VIDEO v2 CHUNK] 청크 처리
- **pass-1**: 위 Seg(Cell 5) + 안정 bbox(Cell 5.5) 는 전 프레임 1회 실행 (안정 bbox 가 전역 percentile 이라 청크로 쪼개면 안 됨).
- **청크**: 아래 `process_chunk` 가 CHUNK 단위로 crop→Depth→anchor→tail/Normal→restore 후 디스크(`work/video/*.npz`, `work/out/*.npy`)에 내리고 RAM 해제.
- 모델은 Depth/Normal 만 청크마다 reload(Seg 는 pass-1 1회). 검증된 per-frame 바디는 정적 셀에서 그대로 옮겼고 루프 대상만 `FRAMES→chunk_frames`.

In [ ]:
# === [VIDEO v2 CHUNK] 청크 처리 함수: crop -> Depth -> anchor -> tail/Normal -> restore+저장 ===
# 검증된 per-frame 바디는 원본 셀(6 crop, 7 Depth, 8 anchor, 9 Normal, 10 restore)에서
# 그대로 가져와 들여쓰기만 했고, 루프 대상만 FRAMES -> chunk_frames 로 바꿨다.
# 모델(Depth/Normal)은 청크마다 load->처리->del 한다(Seg 는 pass-1 에서 1회). 청크 끝나면 CROP 폐기.
import os, gc, numpy as np, torch, cv2

os.makedirs("work/out", exist_ok=True)
os.makedirs("work/video", exist_ok=True)
os.makedirs("work/crop", exist_ok=True)
os.makedirs("work/mask", exist_ok=True)

# full-frame 복귀 (원본 cell 10 의 to_full 그대로)
def to_full(zc, ib, H, W):
    x0, y0, x1, y1 = ib
    full = np.zeros((H, W), np.float32)
    sx0, sy0 = max(0, x0), max(0, y0)
    sx1, sy1 = min(W, x1), min(H, y1)
    dx, dy = sx0 - x0, sy0 - y0
    if sx1 > sx0 and sy1 > sy0:
        full[sy0:sy1, sx0:sx1] = zc[dy:dy + (sy1 - sy0), dx:dx + (sx1 - sx0)]
    return full

def _fid(f):
    try: return int(f["fid"])
    except Exception: return f["fid"]


def process_chunk(chunk_frames):
    CROP = {}   # 청크 로컬 -> 함수 종료시 자동 해제

    # ---------- [6] crop ----------
    for f in chunk_frames:
        rgb, _ = load_rgb(f["rgb"])

        H, W = f["hw"]

        mask = MASKS[f["key"]]

        depth = (
            to_rgb_res(
                np.load(f["depth"]).astype(np.float32),
                W,
                H,
                nearest=True,
            )
            if f["depth"]
            else None
        )

        conf = (
            to_rgb_res(
                np.load(f["conf"]).astype(np.float32),
                W,
                H,
                nearest=True,
            )
            if f["conf"]
            else None
        )

        margin = (
            0.25
            if f["pose"] == "TPOSE"
            else 0.18
        )

        box = pad_bbox_3_4(
            *f["bbox"],
            margin=margin,
        )

        rgb_c, ib = crop_pad(
            rgb,
            box,
            fill=128,
        )

        mask_c = crop_pad(
            mask.astype(np.uint8),
            box,
            fill=0,
        )[0].astype(bool)

        depth_c = (
            crop_pad(
                depth,
                box,
                fill=0,
            )[0]
            if depth is not None
            else None
        )

        conf_c = (
            crop_pad(
                conf,
                box,
                fill=0,
            )[0]
            if conf is not None
            else None
        )

        CROP[f["key"]] = dict(
            rgb=rgb_c,
            mask=mask_c,
            depth=depth_c,
            conf=conf_c,
            ibox=ib,
        )

        if _dbg(f): cv2.imwrite(
            f"work/crop/{f['key']}_crop.png",
            cv2.cvtColor(
                rgb_c,
                cv2.COLOR_RGB2BGR,
            ),
        )

        ch, cw = rgb_c.shape[:2]

        print(
            f"{f['key']}"
            f" | crop={cw}x{ch}"
            f" | ratio={cw / ch:.3f}"
            f" | margin={margin:.2f}"
        )

    # ---------- [7] Sapiens Depth 1B ----------
    DEPTH_REPO = "facebook/sapiens-depth-1b-torchscript"

    DEPTH_FILE = (
        "sapiens_1b_render_people_epoch_88_"
        "torchscript.pt2"
    )

    DEPTH_URL = (
        f"https://huggingface.co/{DEPTH_REPO}/resolve/main/{DEPTH_FILE}"
    )

    DEPTH_PATH = f"{MODEL_DIR}/sapiens_depth_1b.pt2"

    os.makedirs(MODEL_DIR, exist_ok=True)

    ensure_model(
        DEPTH_URL,
        DEPTH_PATH,
        min_gb=3.5,
    )

    dmodel = torch.jit.load(
        DEPTH_PATH,
    ).cuda().eval()


    @torch.no_grad()
    def sapiens_depth(rgb_c):
        ch, cw = rgb_c.shape[:2]

        x = cv2.resize(
            rgb_c,
            (WS, HS),
            interpolation=cv2.INTER_LINEAR,
        )

        x = x.astype(np.float32) / 255.0
        x = (x - MEAN) / STD

        x = torch.from_numpy(x)
        x = x.permute(2, 0, 1)
        x = x.unsqueeze(0).float().cuda()

        out = unwrap_tensor(dmodel(x))

        out = (
            out.squeeze()
            .float()
            .cpu()
            .numpy()
        )

        if out.ndim != 2:
            raise RuntimeError(
                f"Depth 출력 shape 오류: {out.shape}"
            )

        return cv2.resize(
            out,
            (cw, ch),
            interpolation=cv2.INTER_LINEAR,
        )


    for f in chunk_frames:
        CROP[f["key"]]["sap"] = sapiens_depth(
            CROP[f["key"]]["rgb"]
        )

        print(
            f"{f['key']}: "
            f"sap={CROP[f['key']]['sap'].shape}"
        )

    del dmodel
    torch.cuda.empty_cache()

    # ---------- [8] 사람 LiDAR anchor ----------
    MIN_ANCHOR=300   # 이보다 적으면 a,b 불안정 경고
    FORCE_MODE="depth"   # [VIDEO v2] depth 고정(mode flip 방지). 자동판별로 되돌리려면 None.
    for f in chunk_frames:
        C=CROP[f['key']]; s=C['sap']; m=C['mask']
        if C['depth'] is None or C['conf'] is None:
            print(f"{f['key']}: depth/conf 없음 → skip"); continue
        m_er=cv2.erode(m.astype(np.uint8),np.ones((5,5),np.uint8),1).astype(bool)
        valid=m_er&(C['depth']>1e-6)&np.isfinite(C['depth'])&(C['conf']>=2)
        relaxed=False
        if valid.sum()<MIN_ANCHOR:   # conf==2 부족 → conf>=1 완화
            valid=m_er&(C['depth']>1e-6)&np.isfinite(C['depth'])&(C['conf']>=1); relaxed=True
        n_anc=int(valid.sum())
        if n_anc<50:
            print(f"{f['key']}: ⛔ anchor 점 {n_anc}개 — 너무 적음. 촬영 거리 가깝게/conf 점검. skip"); continue
        p=anchor_auto(s[valid].astype(np.float64), C['depth'][valid].astype(np.float64), force_mode=FORCE_MODE)
        z=apply_anchor(s,p); z[~m]=0.0; C['zmetric']=z; C['anchor']=p
        zf=z[m&(z>0)]
        warn=[]
        if n_anc<MIN_ANCHOR: warn.append(f'점부족({n_anc})')
        if relaxed: warn.append('conf>=1완화')
        if p['inlier']<0.6: warn.append(f"inlier낮음({p['inlier']*100:.0f}%)")
        if p['rel']>0.05: warn.append(f"잔차높음({p['rel']*100:.1f}%)")
        flag=('⚠ '+', '.join(warn)) if warn else '✓'
        REPORT.append(dict(key=f['key'],pose=f['pose'],view=f['view'],n_anchor=n_anc,relaxed=relaxed,
                           **{k:p[k] for k in ['mode','a','b','rel','inlier']},
                           z_med=float(np.median(zf)),z_min=float(zf.min()),z_max=float(zf.max()),flag=flag))
        print(f"{f['key']}: {p['mode']:9s} rel={p['rel']*100:.2f}% inlier={p['inlier']*100:.0f}% N={n_anc} z_med={np.median(zf):.3f}m  {flag}")

    # ---------- [9] tail 제거 + Normal refine ----------
    DO_NORMAL_REFINE = True

    # --- tail 제거 임계 (필요시 조정) ---
    GRAZE_DEG = 75     # 작을수록 공격적(70~80). 시선과 이 각도 넘게 기운 면 제거
    DISC_REL  = 0.03   # 이웃 depth 점프 임계(깊이 대비). 팔-몸통 과제거되면 0.05
    ERODE_PX  = 2      # 실루엣 띠 제거 px

    if DO_NORMAL_REFINE:

        NORMAL_REPO = "facebook/sapiens-normal-1b-torchscript"
        NORMAL_FILE = (
            "sapiens_1b_normal_render_people_epoch_115_torchscript.pt2"
        )
        NORMAL_URL  = f"https://huggingface.co/{NORMAL_REPO}/resolve/main/{NORMAL_FILE}"
        NORMAL_PATH = f"{MODEL_DIR}/sapiens_normal_1b.pt2"

        ensure_model(NORMAL_URL, NORMAL_PATH, min_gb=3.5)
        nmodel = torch.jit.load(NORMAL_PATH).cuda().eval()

        @torch.no_grad()
        def sap_normal(rgb_c):
            ch, cw = rgb_c.shape[:2]
            x = cv2.resize(rgb_c, (WS, HS), interpolation=cv2.INTER_LINEAR)
            x = x.astype(np.float32) / 255.0
            x = (x - MEAN) / STD
            x = torch.from_numpy(x).permute(2, 0, 1).unsqueeze(0).float().cuda()
            n = unwrap_tensor(nmodel(x))
            if n.ndim == 4:
                n = n[0].permute(1, 2, 0)
            elif n.ndim == 3:
                n = n.permute(1, 2, 0)
            else:
                raise RuntimeError(f"Normal 출력 shape 오류: {tuple(n.shape)}")
            n = n.float().cpu().numpy()
            n = cv2.resize(n, (cw, ch), interpolation=cv2.INTER_LINEAR)
            return n / (np.linalg.norm(n, axis=-1, keepdims=True) + 1e-8)

        # ---------- tail(flying pixel) 제거 ----------
        def clean_flying(z, mask, fx, fy, cx, cy,
                         rel=DISC_REL, erode_px=ERODE_PX,
                         normal=None, graze_deg=GRAZE_DEG):
            M = mask & (z > 0)
            keep = M.copy()
            # (1) 실루엣 띠 제거 — tail의 뿌리
            if erode_px > 0:
                k = 2 * erode_px + 1
                keep &= cv2.erode(M.astype(np.uint8),
                                  np.ones((k, k), np.uint8), 1).astype(bool)
            # (2) depth discontinuity — 날카로운 점프
            jump = np.zeros_like(z)
            for dy, dx in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                sh = np.roll(np.roll(z, dy, 0), dx, 1)
                vd = np.roll(np.roll(M, dy, 0), dx, 1)
                jump = np.maximum(jump, np.where(vd, np.abs(z - sh), 0))
            keep &= jump < rel * np.clip(z, 1e-3, None)
            # (3) grazing — 시선과 크게 기운 면 (curtain tail 주범)
            if normal is not None:
                h, w = z.shape
                uu, vv = np.meshgrid(np.arange(w), np.arange(h))
                r = np.stack([(uu - cx) / fx, (vv - cy) / fy, np.ones_like(z)], -1)
                r /= np.linalg.norm(r, axis=-1, keepdims=True)
                keep &= np.abs((normal * r).sum(-1)) > np.cos(np.deg2rad(graze_deg))
            z2 = z.copy()
            z2[~keep] = 0.0
            return keep, z2

        # ---------- perspective screened-Poisson refine ----------
        def refine(z, n, mask, fx, fy, cx, cy, lam=0.1, iters=800):
            h, w = z.shape
            uu, vv = np.meshgrid(np.arange(w), np.arange(h))
            a = (uu - cx) / fx
            c = (vv - cy) / fy
            nx, ny, nz = n[..., 0], n[..., 1], n[..., 2]
            D = nx * a + ny * c + nz
            D = np.where(np.abs(D) < 1e-3, np.sign(D) * 1e-3 + 1e-9, D)
            Zu = -z * nx / (fx * D)
            Zv = -z * ny / (fy * D)
            reliable = np.abs(nz) > 0.3
            M = mask & (z > 0)
            er = cv2.erode(M.astype(np.uint8), np.ones((5, 5), np.uint8), 1).astype(bool)
            wd = np.where(M & ~er, lam * 6, lam)
            z0 = z.copy()
            d = z.copy()
            for _ in range(iters):
                num = np.zeros_like(d)
                den = np.zeros_like(d)
                lk = M[:, :-1] & M[:, 1:] & reliable[:, :-1]
                num[:, :-1] += np.where(lk, d[:, 1:] - Zu[:, :-1], 0); den[:, :-1] += lk
                num[:, 1:]  += np.where(lk, d[:, :-1] + Zu[:, :-1], 0); den[:, 1:]  += lk
                lv = M[:-1, :] & M[1:, :] & reliable[:-1, :]
                num[:-1, :] += np.where(lv, d[1:, :] - Zv[:-1, :], 0); den[:-1, :] += lv
                num[1:, :]  += np.where(lv, d[:-1, :] + Zv[:-1, :], 0); den[1:, :]  += lv
                d = np.where(M, (num + wd * z0) / (den + wd + 1e-9), z0)
            d[~mask] = 0.0
            return d

        for f in chunk_frames:
            C = CROP[f["key"]]
            if "zmetric" not in C:
                continue

            H, W = f["hw"]
            x0, y0, _, _ = C["ibox"]
            fx, fy, cx, cy = get_intr(f["key"], W, H)
            cxc, cyc = cx - x0, cy - y0   # crop 기준 intrinsic

            nrm = sap_normal(C["rgb"])
            C["normal"] = nrm.astype(np.float32)   # [VIDEO] Phase 4 재-refine 연료

            # tail 제거 → 깨끗한 mask/depth
            before = int((C["mask"] & (C["zmetric"] > 0)).sum())
            keep, z_clean = clean_flying(C["zmetric"], C["mask"], fx, fy, cxc, cyc, normal=nrm)
            after = int(keep.sum())

            # 깨끗해진 geometry 위에서 refine
            C["mask"]    = keep
            C["zmetric"] = refine(z_clean, nrm, keep, fx, fy, cxc, cyc)

            print(f"{f['key']}: tail 제거 {before}→{after} "
                  f"({100*(before-after)/max(before,1):.1f}% 컷) + refine")

        del nmodel
        torch.cuda.empty_cache()
        print("✓ tail 제거 + Normal refine 완료")

    # ---------- [10] full-frame 복귀 + 키/너비 측정 + npy/npz 저장 ----------
    rep = {r["key"]: r for r in REPORT}
    for f in chunk_frames:
        k = f["key"]; C = CROP.get(k)
        if C is None or "zmetric" not in C:
            continue
        H, W = f["hw"]
        zfull = to_full(C["zmetric"], C["ibox"], H, W)
        mf = MASKS[k]
        fx, fy, cx, cy = get_intr(k, W, H)
        h  = measure_height_m(zfull, mf, fx, fy, cx, cy)
        wd = measure_width_m(zfull, mf, fx, fy, cx, cy)
        np.save(f"work/out/{k}_person_metric.npy", zfull)

        r = rep.get(k, {})
        payload = dict(
            s_relative=C["sap"].astype(np.float32),
            zmetric_crop=C["zmetric"].astype(np.float32),
            mask_crop=C["mask"], ibox=np.array(C["ibox"], np.int32),
            bbox=np.array(f["bbox"], np.int32),
            fid=str(f["fid"]),
            height_m=np.float32(h if h is not None else np.nan),
            width_m=np.float32(wd if wd is not None else np.nan),
            a_raw=np.float32(r.get("a", np.nan)), b_raw=np.float32(r.get("b", np.nan)),
            mode=str(r.get("mode", "")), n_anchor=np.int32(r.get("n_anchor", 0)),
            rel=np.float32(r.get("rel", np.nan)), inlier=np.float32(r.get("inlier", np.nan)))
        if SAVE_NORMAL and "normal" in C:
            payload["normal"] = C["normal"]
        np.savez_compressed(f"work/video/{k}.npz", **payload)

        HT.append(h); WT.append(wd); KEYS.append(k); FIDS.append(_fid(f))

    # ---------- 청크 RAM 해제 ----------
    CROP.clear(); del CROP
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# === [VIDEO v2 CHUNK] 청크 드라이버 -- pass-1(Seg+stable bbox) 뒤에 실행 ===
import json, gc, torch, os
assert "process_chunk" in globals(), "먼저 process_chunk 정의 셀을 실행하세요."
assert "MASKS" in globals() and len(FRAMES) > 0, "Seg(Cell 5)·stable bbox(Cell 5.5) 먼저 실행."

REPORT, HT, WT, KEYS, FIDS = [], [], [], [], []     # 전 청크 누적(전역)
# [colab_kit] 재시작 이어하기: 이미 work/video/{key}.npz 가 있는 프레임은 건너뜀
TODO = [f for f in FRAMES if not os.path.exists(f"work/video/{f['key']}.npz")]
print(f"[resume] 완료 {len(FRAMES)-len(TODO)} / 남음 {len(TODO)}")
chunks = [TODO[i:i+CHUNK] for i in range(0, len(TODO), CHUNK)]
print(f"[VIDEO v2] 총 {len(FRAMES)} 프레임 -> {len(chunks)} 청크 (CHUNK={CHUNK})")

for ci, chunk in enumerate(chunks, 1):
    fids = [f["fid"] for f in chunk]
    print(f"\n===== 청크 {ci}/{len(chunks)} | {len(chunk)} 프레임 | fid {fids[0]}~{fids[-1]} =====")
    process_chunk(chunk)                            # crop->Depth->anchor->tail/Normal->restore+저장
    gc.collect(); torch.cuda.empty_cache()
    print(f"[청크 {ci}] 완료 | 누적 person_metric {len(KEYS)} / REPORT {len(REPORT)}")

json.dump(REPORT, open("work/out/anchor_report.json","w"), indent=2, ensure_ascii=False)
print(f"\n✓ 전 청크 완료 | person_metric {len(KEYS)}개 저장 | anchor_report.json")
print("-> 마지막 '합산' 셀에서 250 전체 anchor_series + 시간축 리포트 생성")

## 10.6 / 10.7 [VIDEO] 후처리 연료 + 시간축 검증
`s_relative`(재anchor), `normal`(재refine), anchor 시계열을 **raw로** 저장하고(Phase 3/4 연료), 프레임축 키 std를 본다.

In [ ]:
# === [VIDEO v2 CHUNK] 최종 합산: 전 청크 npz를 fid 순으로 읽어 anchor_series + 시간축 리포트 ===
#   디스크(work/video/*.npz) 기반 -> 중간에 커널이 끊겨도 저장된 청크까지 합산 가능.
import numpy as np, glob, os, json
import matplotlib.pyplot as plt
MY_HEIGHT_M = 1.81        # 실제 신장(맨발) 알면 입력 -> 절대 bias 출력. 모르면 None

paths = [p for p in sorted(glob.glob("work/video/*.npz"))
         if os.path.basename(p) != "anchor_series.npz"]
recs = []
for p in paths:
    d = np.load(p, allow_pickle=True)
    if "fid" not in d.files:                          # 구버전 npz 안전 스킵
        continue
    fid = str(d["fid"])
    try: fkey = int(fid)
    except Exception: fkey = fid
    recs.append((fkey, os.path.basename(p)[:-4],
                 float(d["a_raw"]), float(d["b_raw"]), int(d["n_anchor"]),
                 float(d["height_m"]), float(d["width_m"])))
recs.sort(key=lambda r: r[0])                         # fid 순

keys = np.array([r[1] for r in recs])
a    = np.array([r[2] for r in recs], np.float32)
b    = np.array([r[3] for r in recs], np.float32)
nn   = np.array([r[4] for r in recs], np.int32)
H    = np.array([r[5] for r in recs], np.float64)
Wd   = np.array([r[6] for r in recs], np.float64)

np.savez("work/video/anchor_series.npz", a=a, b=b, n_anchor=nn, keys=keys)   # raw 그대로(평활 안 함)

Hm = H[np.isfinite(H)]
report = dict(n_frames=len(recs),
              a_std=float(np.nanstd(a)) if len(a) else None,
              b_std=float(np.nanstd(b)) if len(b) else None,
              anchor_bad_rate=float((nn < 300).mean()) if len(nn) else None,
              height_mean=float(Hm.mean()) if len(Hm) else None,
              height_std_cm=float(Hm.std() * 100) if len(Hm) else None,
              height_range_cm=float((Hm.max() - Hm.min()) * 100) if len(Hm) else None)
if MY_HEIGHT_M and len(Hm):
    report["abs_bias_cm"] = float((Hm.mean() - MY_HEIGHT_M) * 100)
json.dump(report, open("work/video/flicker_report.json", "w"), indent=2, ensure_ascii=False)

print(f"[합산] {len(recs)} 프레임  -> work/video/flicker_report.json")
if len(recs):
    print(f"[anchor] a std={report['a_std']:.4f}  b std={report['b_std']:.4f}  "
          f"| anchor 불량률(n<300)={report['anchor_bad_rate'] * 100:.1f}%")
if len(Hm):
    print(f"[키] mean={report['height_mean']:.3f} m  std={report['height_std_cm']:.2f} cm  "
          f"range={report['height_range_cm']:.1f} cm  (N={len(Hm)})")
    print("-> 보행 중 std <= +-2cm 면 anchor 안정. 크면 Phase4 anchor 평활(EMA/칼만).")
    if MY_HEIGHT_M:
        print(f"   절대 bias = {report['abs_bias_cm']:+.1f} cm (신발/머리카락 실루엣 여유 포함)")
    plt.figure(figsize=(10, 3))
    plt.plot(H, lw=1.2)
    plt.axhline(Hm.mean(), ls="--", c="g", label=f"mean {Hm.mean():.3f}m")
    plt.fill_between(range(len(H)), Hm.mean() - 0.02, Hm.mean() + 0.02,
                     color="g", alpha=0.12, label="+-2cm")
    plt.xlabel("frame (fid 순)"); plt.ylabel("height (m)"); plt.legend(loc="upper right")
    plt.title(f"per-frame height  std={report['height_std_cm']:.2f}cm  (N={len(recs)})")
    plt.tight_layout(); plt.savefig("work/video/height_temporal.png", dpi=110); plt.show()
else:
    print("측정된 키가 없음 -- 앞 단계 확인")

## 시각화 (NB2)

In [ ]:
#@title 시각화 — NB2 결과 확인 (viz/NB2/  + viz/videos/)
FIDS = ""   #@param {type:"string"}
N_EVEN = 4           #@param {type:"integer"}
# 셀 0 을 안 돌렸어도 혼자 동작하도록 (드라이브 마운트 + 세션 설정)
import os, sys, importlib
if "walkcfg" not in globals():
    ROOT = "/content/drive/MyDrive/walking_0729"   # 셀 0 과 같게
    SESSION = "0729_full_3kmh"                     # 셀 0 과 같게
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["WALK_ROOT"] = ROOT; os.environ["WALK_SESSION"] = SESSION
    sys.path.insert(0, f"{ROOT}/scripts"); import walkcfg
print("세션:", walkcfg.SESS)
MAKE_VIDEO = True    #@param {type:"boolean"}
FPS = 15             #@param {type:"number"}
!pip -q install imageio-ffmpeg
!cd "{ROOT}/scripts" && python viz_stages.py --nb 2 --fids {FIDS} --n {N_EVEN}
if MAKE_VIDEO:
    !cd "{ROOT}/scripts" && python viz_video.py --nb 2 --fps {FPS}
from IPython.display import Image, display
import glob, os
for p in sorted(glob.glob(f"{walkcfg.SESS}/viz/NB2/*.png")):
    print(os.path.basename(p)); display(Image(p, width=1100))
from IPython.display import Video
_mp4 = f"{walkcfg.SESS}/viz/videos/NB2_depth.mp4"
if os.path.exists(_mp4):
    print("영상:", _mp4, f"{os.path.getsize(_mp4)/1e6:.1f} MB  (노트북에 넣어 재생 — 확인 후 출력 지우기 권장)")
    display(Video(_mp4, embed=True, width=1100))